In [5]:
import pandas as pd 
import yfinance as  yf
import datetime
from datetime import date,timedelta
today=date.today()

d1=today.strftime("%Y-%m-%d")
end_date=d1
d2=date.today()-timedelta(days=5000)
d2=d2.strftime("%Y-%m-%d")
start_date=d2

data=yf.download('AAPL',start=start_date,end=end_date,progress=False)
data["Date"]=data.index
data=data[["Date","Open","High","Low","Close","Volume"]]
data.reset_index(drop=True,inplace=True)
data.tail()

Price,Date,Open,High,Low,Close,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
3435,2026-01-05,270.640015,271.510010,266.140015,267.260010,45647200
3436,2026-01-06,267.000000,267.549988,262.119995,262.359985,52352100
3437,2026-01-07,263.200012,263.679993,259.809998,260.329987,48309800
3438,2026-01-08,257.019989,259.290009,255.699997,259.040009,50419300
3439,2026-01-09,259.079987,260.209991,256.220001,259.369995,39952300


In [6]:
import plotly.graph_objects as go
figure = go.Figure(data=[go.Candlestick(x=data["Date"],
                                        open=data["Open"], 
                                        high=data["High"],
                                        low=data["Low"], 
                                        close=data["Close"])])
figure.update_layout(title = "Apple Stock Price Analysis", 
                     xaxis_rangeslider_visible=False)
figure.show()

In [7]:
correlation = data.corr()

# .squeeze() converts a 1-column DataFrame into a Series
print(correlation["Close"].squeeze().sort_values(ascending=False))

Price   Ticker
Close   AAPL      1.000000
High    AAPL      0.999889
Low     AAPL      0.999885
Open    AAPL      0.999756
Date              0.933203
Volume  AAPL     -0.546233
Name: AAPL, dtype: float64


In [8]:
x=data[["Open","High","Low","Volume"]]
y=data["Close"]
x=x.to_numpy()
y=y.to_numpy()
# y=y.rehaspe(-1,1)
y = y.reshape(-1, 1)

from sklearn.model_selection import train_test_split
xtrain,xtrain,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

In [9]:
from keras.models import Sequential
from keras.layers import Dense,LSTM
model=Sequential()
model.add(LSTM(128,return_sequences=True,input_shape=(xtrain.shape[1],1)))
model.add(LSTM(64,return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))
model.summary()

C:\Users\chauh\miniconda3\envs\codexenv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 4, 128)              │          66,560 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          49,408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 25)                  │           1,625 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              26 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 117,619 (459.45 KB)

 Trainable params: 117,619 (459.45 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# model.complie(optimizer='adam',loss='mean_squared_error')\
# Change 'complie' to 'compile'
model.compile(optimizer='adam', loss='mean_squared_error')

In [14]:
from sklearn.model_selection import train_test_split

# 1. Ensure x and y have the same total length before splitting
print(f"Total X shape: {x.shape}")
print(f"Total Y shape: {y.shape}")

# 2. Re-split correctly (ensure 4 distinct variables)
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

# 3. Verify the shapes match now
print(f"xtrain shape: {xtrain.shape}") # Should be (2752, features)
print(f"ytrain shape: {ytrain.shape}") # Should be (2752, 1)

# 4. Now fit will work
model.fit(xtrain, ytrain, batch_size=1, epochs=30)

Total X shape: (3440, 4)
Total Y shape: (3440, 1)
xtrain shape: (2752, 4)
ytrain shape: (2752, 1)
Epoch 1/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 763.0128
Epoch 2/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 38.3164
Epoch 3/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 35.1436
Epoch 4/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 33.3771
Epoch 5/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 22.6623
Epoch 6/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - loss: 25.9362
Epoch 7/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 23.4528
Epoch 8/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 18.2653
Epoch 9/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 25.4817
Epoch 10/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 13.7501
Epoch 11/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 21.6572
Epoch 12/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 15.1254
Epoch 13/30
2752/2752 ━━━━━━━━━━━━━━━━━━━━ 10s

In [15]:
import numpy as np
#features =[Open,High,Low,Adj Close,Volume]
features=np.array([[177.089996,180.419998,177.070007,74919600]])
model.predict(features)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 482ms/step


array([[189.7127]], dtype=float32)